In [1]:
import json, os
from pathlib import Path
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm")
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = PROJECT / "checkpoints" / "qwen1.5b-qlora-v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

torch: 2.11.0+cu128 cuda: True


In [36]:
SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.
- "Except X" or "aside from X" followed by negative language about the rest means ONLY X is favored: output [X].
- Ordinal references map A=first, B=second, C=third, D=fourth.

Output ONLY the JSON. No explanation, no prose."""

def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

train_raw = load_jsonl(PROJECT / "data" / "train.jsonl")
test_raw  = load_jsonl(PROJECT / "data" / "test.jsonl")

def to_chat(ex):
    user = f'Options: {ex["options"]}\nUser: {ex["utterance"]}'
    assistant = json.dumps(ex["label"])
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user},
            {"role": "assistant", "content": assistant},
        ]
    }

train_ds = Dataset.from_list([to_chat(e) for e in train_raw])
test_ds  = Dataset.from_list([to_chat(e) for e in test_raw])
print("train:", len(train_ds), "test:", len(test_ds))
print("sample:", train_ds[0])

train: 1278 test: 210
sample: {'messages': [{'role': 'system', 'content': 'You are a parser that converts a user\'s natural language response into a subset of the shown options.\n\nThe user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.\n\nOutput format (JSON only, nothing else):\n- A JSON list of the favored labels, e.g. ["A", "B"]\n- [] if the user explicitly rejects ALL options ("none of these", "all wrong")\n- "*" if the utterance is off-topic OR expresses no usable preference ("I don\'t know", "they all look the same", "I love football")\n\nRules:\n- Any positive signal about an option means it goes in the list.\n- "X is better than Y" endorses only X, not Y.\n- "X and Y are both good, X is better" endorses both X and Y.\n- Negations like "not D" or "anything but B" mean the remaining options go in the list.\n- Questions like "is it A?" are treated as tentative endorsem

In [37]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
model = prepare_model_for_kbit_training(model)
print("model loaded. VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

model loaded. VRAM (GB): 5.63982336


In [38]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [39]:
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,
    packing=False,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

In [40]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.054707,0.053268,0.058578,422606.000000,0.988745
2,0.046507,0.047874,0.047362,845212.000000,0.989855
3,0.044016,0.047361,0.045534,1267818.000000,0.990051


TrainOutput(global_step=240, training_loss=0.1613874563326438, metrics={'train_runtime': 1293.267, 'train_samples_per_second': 2.965, 'train_steps_per_second': 0.186, 'total_flos': 1.0199309446146048e+16, 'train_loss': 0.1613874563326438, 'epoch': 3.0})

In [41]:
for n, p in model.named_parameters():
    if p.requires_grad:
        print(n, p.dtype)
        break

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16


In [43]:
from peft import PeftModel

# Base model in 4-bit
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)

# Load adapter
adapter_path = str(OUTPUT_DIR / "checkpoint-219")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter loaded from C:\Users\shlok\projects\ddp-llm\checkpoints\qwen1.5b-qlora-v1\checkpoint-219


In [44]:
import gc, torch
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("VRAM after cleanup:", torch.cuda.memory_allocated() / 1e9, "GB")

VRAM after cleanup: 4.266226688 GB


In [45]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
adapter_path = str(OUTPUT_DIR / "checkpoint-219")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)
print("VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter loaded from C:\Users\shlok\projects\ddp-llm\checkpoints\qwen1.5b-qlora-v1\checkpoint-219
VRAM (GB): 4.95252992


In [46]:
def generate_ft(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

test_cases = [
    ("A and C look good", '["A", "C"]'),
    ("not D", '["A", "B", "C"]'),
    ("I don't know", '"*"'),
    ("none of these work", '[]'),
    ("A is better than B", '["A"]'),
    ("what's for lunch", '"*"'),
    ("hate all of them", '[]'),
    ("A great, B bad, C great, D bad", '["A", "C"]'),
]

for utt, expected in test_cases:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: ["A", "B", "C", "D"]\nUser: {utt}'},
    ]
    got = generate_ft(msgs)
    match = "✓" if got.strip() == expected else "✗"
    print(f"{match} {utt!r}\n   expected: {expected}\n   got:      {got}\n")

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✓ 'A and C look good'
   expected: ["A", "C"]
   got:      ["A", "C"]

✓ 'not D'
   expected: ["A", "B", "C"]
   got:      ["A", "B", "C"]

✓ "I don't know"
   expected: "*"
   got:      "*"

✓ 'none of these work'
   expected: []
   got:      []

✓ 'A is better than B'
   expected: ["A"]
   got:      ["A"]

✓ "what's for lunch"
   expected: "*"
   got:      "*"

✓ 'hate all of them'
   expected: []
   got:      []

✓ 'A great, B bad, C great, D bad'
   expected: ["A", "C"]
   got:      ["A", "C"]



In [47]:
from tqdm import tqdm

def parse_output(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in ["A","B","C","D"] for x in parsed)):
        return parsed
    return None

def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False

test_examples = load_jsonl(PROJECT / "data" / "test.jsonl")

results = []
for ex in tqdm(test_examples):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {ex["options"]}\nUser: {ex["utterance"]}'},
    ]
    raw = generate_ft(msgs)
    parsed = parse_output(raw)
    correct = parsed is not None and labels_equal(parsed, ex["label"])
    results.append({
        "utterance": ex["utterance"],
        "category": ex.get("category", "unknown"),
        "gold": ex["label"],
        "raw": raw,
        "parsed": parsed,
        "correct": correct,
        "valid_format": parsed is not None,
    })

n = len(results)
n_valid = sum(r["valid_format"] for r in results)
n_correct = sum(r["correct"] for r in results)
print(f"\nformat validity: {n_valid}/{n} = {n_valid/n:.1%}")
print(f"exact match:     {n_correct}/{n} = {n_correct/n:.1%}")

from collections import defaultdict
by_cat = defaultdict(lambda: [0, 0])
for r in results:
    by_cat[r["category"]][0] += 1
    by_cat[r["category"]][1] += int(r["correct"])
print("\nper category:")
for cat, (total, correct) in sorted(by_cat.items()):
    print(f"  {cat}: {correct}/{total} = {correct/total:.1%}")

100%|██████████| 210/210 [04:24<00:00,  1.26s/it]


format validity: 209/210 = 99.5%
exact match:     197/210 = 93.8%

per category:
  comparative: 24/26 = 92.3%
  mixed_sentiment: 26/28 = 92.9%
  multi_positive: 12/13 = 92.3%
  negation: 53/61 = 86.9%
  off_topic: 22/22 = 100.0%
  reject_all: 29/29 = 100.0%
  single_positive: 9/9 = 100.0%
  uncertainty: 22/22 = 100.0%


In [48]:
print("=== FAILURES ===")
for r in results:
    if not r["correct"]:
        print(f"[{r['category']}] {r['utterance']!r}")
        print(f"   gold:   {r['gold']}")
        print(f"   parsed: {r['parsed']}")
        print(f"   raw:    {r['raw']!r}")
        print()

=== FAILURES ===
[negation] 'except B, all are off'
   gold:   ['B']
   parsed: ['A', 'C', 'D']
   raw:    '["A", "C", "D"]'

[negation] 'everyone except B is a miss'
   gold:   ['B']
   parsed: ['A', 'C', 'D']
   raw:    '["A", "C", "D"]'

[negation] 'the first, second, and third are all wrong'
   gold:   ['D']
   parsed: None
   raw:    '["fourth"]'

[negation] 'aside from B, none work'
   gold:   ['B']
   parsed: ['A', 'C', 'D']
   raw:    '["A", "C", "D"]'

[negation] 'except D, all are off'
   gold:   ['D']
   parsed: ['A', 'B', 'C']
   raw:    '["A", "B", "C"]'

[negation] 'the first, third, and fourth are all wrong'
   gold:   ['B']
   parsed: ['A', 'B', 'D']
   raw:    '["A", "B", "D"]'

[comparative] 'B leaves D in the dust'
   gold:   ['B']
   parsed: ['A', 'C']
   raw:    '["A", "C"]'

[negation] 'the fourth is wrong'
   gold:   ['A', 'B', 'C']
   parsed: ['A', 'B', 'D']
   raw:    '["A", "B", "D"]'

[comparative] 'C and D are both in the running, C edges it out'
   gold:   

In [49]:
import gc, torch
try:
    del base, ft_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("VRAM after cleanup:", torch.cuda.memory_allocated() / 1e9, "GB")

VRAM after cleanup: 3.037109248 GB


In [9]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
adapter_path = str(OUTPUT_DIR / "checkpoint-219")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)
print("VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter loaded from C:\Users\shlok\projects\ddp-llm\checkpoints\qwen1.5b-qlora-v1\checkpoint-219
VRAM (GB): 4.038555648


In [10]:
import json
from pathlib import Path
from collections import Counter

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm")
train = [json.loads(l) for l in (PROJECT / "data" / "train.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
test  = [json.loads(l) for l in (PROJECT / "data" / "test.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]

print("train negation:", sum(1 for e in train if e.get("category") == "negation"))
print("test  negation:", sum(1 for e in test  if e.get("category") == "negation"))

train negation: 252
test  negation: 42


In [11]:
neg_train = [e for e in train if e.get("category") == "negation"]

def has_except_all_wrong(u):
    u = u.lower()
    return "except" in u and any(w in u for w in ["all wrong", "all off", "everything is", "everything's", "the rest are wrong", "all are off", "everything else is wrong", "everything else is off"])

def has_ordinal(u):
    u = u.lower()
    ordinals = ["first one", "second one", "third one", "fourth one", "the first", "the second", "the third", "the fourth"]
    return any(o in u for o in ordinals)

print(f"total negation train: {len(neg_train)}")
print(f"  'except X, all wrong'-shape: {sum(1 for e in neg_train if has_except_all_wrong(e['utterance']))}")
print(f"  ordinal references: {sum(1 for e in neg_train if has_ordinal(e['utterance']))}")

print("\nsample 'except X'-ish examples:")
for e in neg_train:
    if "except" in e["utterance"].lower():
        print(f"  {e['utterance']!r} -> {e['label']}")

total negation train: 252
  'except X, all wrong'-shape: 2
  ordinal references: 6

sample 'except X'-ish examples:
  'cut all except A' -> ['A']
  'remove all except D' -> ['D']
  'remove everything except A' -> ['A']
  'except for D everything is bad' -> ['D']
  'anything except D' -> ['A', 'B', 'C']
  'all except C and D' -> ['A', 'B']
  'all of them except none, A B C and D all decent' -> ['A', 'B', 'C', 'D']
  'except for B' -> ['A', 'C', 'D']
  'everything except B and D is a candidate' -> ['A', 'C']
  'everything except D is a candidate' -> ['A', 'B', 'C']
  'everything except A and C is a candidate' -> ['B', 'D']
  "except A they're all wrong" -> ['A']
  'everything except B is fine' -> ['A', 'C', 'D']
  'eliminate everything except C' -> ['C']


In [12]:
import json
from pathlib import Path
from collections import Counter

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm")
synth = [json.loads(l) for l in (PROJECT / "data" / "synthetic_raw.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"total in synthetic_raw: {len(synth)}")
print("by category:", Counter(e.get("category", "unknown") for e in synth))

total in synthetic_raw: 1496
by category: Counter({'negation': 426, 'mixed_sentiment': 200, 'reject_all': 200, 'comparative': 180, 'uncertainty': 150, 'off_topic': 150, 'multi_positive': 80, 'single_positive': 60, 'unknown': 50})


In [16]:
import json
from pathlib import Path

PATH = Path(r"C:\Users\shlok\projects\ddp-llm\data\synthetic_raw.jsonl")

kept, dropped = 0, 0
lines_out = []
with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        if ex.get("category", "unknown") == "unknown":
            dropped += 1
            continue
        lines_out.append(line)
        kept += 1

with open(PATH, "w", encoding="utf-8") as f:
    f.writelines(lines_out)

print(f"kept {kept}, dropped {dropped}")

kept 1446, dropped 50


In [15]:
import os; print(os.getcwd())

C:\Users\shlok\projects\ddp-llm\notebooks


In [33]:
import json
from pathlib import Path

PATH = Path(r"C:\Users\shlok\projects\ddp-llm\data\train.jsonl")

except_examples = []
with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        ex = json.loads(line)
        u = ex["utterance"].lower()
        if any(k in u for k in ["except", "aside from", "everyone except", "everything except"]):
            except_examples.append(ex)

print(f"total 'except'-shape examples in train: {len(except_examples)}")
print(f"with single-element label: {sum(1 for e in except_examples if isinstance(e['label'], list) and len(e['label']) == 1)}")
print(f"with multi-element label: {sum(1 for e in except_examples if isinstance(e['label'], list) and len(e['label']) > 1)}")
print("\nsamples:")
for e in except_examples[:8]:
    print(f"  {e['utterance']!r} -> {e['label']}")

total 'except'-shape examples in train: 48
with single-element label: 42
with multi-element label: 6

samples:
  'everything except the third is wrong' -> ['C']
  'everything except B is a mistake' -> ['B']
  'everything except B and D is a candidate' -> ['A', 'C']
  'aside from D, none work' -> ['D']
  'everything except the first is wrong' -> ['A']
  'all of them except none, A B C and D all decent' -> ['A', 'B', 'C', 'D']
  'aside from A, none work' -> ['A']
  'everyone except A is a miss' -> ['A']


In [50]:
import json, os
from pathlib import Path
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm")
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = PROJECT / "checkpoints" / "qwen3b-qlora-v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROJECT / "data" / "train.jsonl"
TEST_PATH = PROJECT / "data" / "test.jsonl"

In [51]:
SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.

Output ONLY the JSON. No explanation, no prose."""

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_raw = load_jsonl(TRAIN_PATH)
test_raw = load_jsonl(TEST_PATH)
print(f"train: {len(train_raw)}, test: {len(test_raw)}")

def to_chat(ex):
    user = f'Options: {ex["options"]}\nUser: {ex["utterance"]}'
    assistant = json.dumps(ex["label"])
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant},
    ]}

train_ds = Dataset.from_list([to_chat(e) for e in train_raw])
test_ds  = Dataset.from_list([to_chat(e) for e in test_raw])

train: 1278, test: 210


In [52]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="cuda",
)
model = prepare_model_for_kbit_training(model)
print(f"VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shlok\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM after load: 5.73 GB


In [53]:
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [54]:
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=False, fp16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,
    packing=False,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
)

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_ds, eval_dataset=test_ds,
    processing_class=tokenizer,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1278 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/210 [00:00<?, ? examples/s]

In [55]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.060615,0.059227,0.065435,358706.000000,0.987261
2,0.051789,0.054397,0.054662,717412.000000,0.987791
3,0.048007,0.054001,0.048622,1076118.000000,0.988087


TrainOutput(global_step=240, training_loss=0.181086765229702, metrics={'train_runtime': 2895.1184, 'train_samples_per_second': 1.324, 'train_steps_per_second': 0.083, 'total_flos': 1.8301691353350144e+16, 'train_loss': 0.181086765229702, 'epoch': 3.0})

In [58]:
import gc
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

VRAM after cleanup: 5.20 GB


In [60]:
from peft import PeftModel

# sanity check what checkpoint dir got saved
print("checkpoints:", os.listdir(OUTPUT_DIR))

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="cuda",
)

# adjust checkpoint number based on what you see above (should be 219 for 1279 train / 16 eff batch * 3 epochs)
CHECKPOINT = "checkpoint-240"
adapter_path = str(OUTPUT_DIR / CHECKPOINT)
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded")

checkpoints: ['checkpoint-160', 'checkpoint-240', 'checkpoint-80', 'README.md']


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

adapter loaded


In [59]:
import os
print(os.listdir(OUTPUT_DIR))

['checkpoint-160', 'checkpoint-240', 'checkpoint-80', 'README.md']


In [61]:
from tqdm.auto import tqdm
from collections import defaultdict

def generate_ft(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

def parse_output(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in ["A","B","C","D"] for x in parsed)):
        return parsed
    return None

def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False

valid, correct = 0, 0
by_cat = defaultdict(lambda: [0, 0])  # [correct, total]
failures = []

for ex in tqdm(test_raw):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {ex["options"]}\nUser: {ex["utterance"]}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw)
    cat = ex.get("category", "unknown")
    by_cat[cat][1] += 1
    if parsed is not None:
        valid += 1
    if parsed is not None and labels_equal(parsed, ex["label"]):
        correct += 1
        by_cat[cat][0] += 1
    else:
        failures.append({"utterance": ex["utterance"], "gold": ex["label"], "parsed": parsed, "raw": raw, "category": cat})

n = len(test_raw)
print(f"\nformat validity: {valid}/{n} = {100*valid/n:.1f}%")
print(f"exact match:     {correct}/{n} = {100*correct/n:.1f}%\n")
print("per category:")
for c in sorted(by_cat):
    ok, tot = by_cat[c]
    print(f"  {c}: {ok}/{tot} = {100*ok/tot:.1f}%")

  0%|          | 0/210 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



format validity: 210/210 = 100.0%
exact match:     210/210 = 100.0%

per category:
  comparative: 26/26 = 100.0%
  mixed_sentiment: 28/28 = 100.0%
  multi_positive: 13/13 = 100.0%
  negation: 61/61 = 100.0%
  off_topic: 22/22 = 100.0%
  reject_all: 29/29 = 100.0%
  single_positive: 9/9 = 100.0%
  uncertainty: 22/22 = 100.0%


In [62]:
print("=== FAILURES ===")
for f in failures:
    print(f"[{f['category']}] {f['utterance']!r}")
    print(f"   gold:   {f['gold']}")
    print(f"   parsed: {f['parsed']}")
    print(f"   raw:    {f['raw']!r}")
    print()

=== FAILURES ===


In [63]:
import json
from pathlib import Path

train = [json.loads(l) for l in open(PROJECT / "data" / "train.jsonl", encoding="utf-8")]
test = [json.loads(l) for l in open(PROJECT / "data" / "test.jsonl", encoding="utf-8")]

train_utts = {e["utterance"].lower().strip() for e in train}
test_utts = {e["utterance"].lower().strip() for e in test}
overlap = train_utts & test_utts
print(f"train: {len(train_utts)}, test: {len(test_utts)}, overlap: {len(overlap)}")
if overlap:
    for u in list(overlap)[:10]:
        print(f"  {u!r}")

train: 1278, test: 210, overlap: 0


In [64]:
MY_UTTERANCE = "eh, only B feels right"
OPTIONS = ["A", "B", "C", "D"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: {OPTIONS}\nUser: {MY_UTTERANCE}'},
]
raw = generate_ft(messages)
parsed = parse_output(raw)
print(f"utterance: {MY_UTTERANCE!r}")
print(f"raw:       {raw!r}")
print(f"parsed:    {parsed}")

utterance: 'eh, only B feels right'
raw:       '["B"]'
parsed:    ['B']


In [65]:
test_utterances = [
    "eh, only B feels right",
    "throw out everything but the third one",
    "hard pass on all four",
    "A? maybe. dunno.",
    "the last two are garbage",
    "B leaves D in the dust",
    "A is as good as C but not better than D",
    "meh whatever pick anything",
    "the second and fourth ones look promising",
    "not really feeling any of them tbh",
    "A is decent, B slightly better, C and D no",
    "give me option C or nothing",
]

for u in test_utterances:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {["A","B","C","D"]}\nUser: {u}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw)
    print(f"{u!r:60s} -> {parsed}")

'eh, only B feels right'                                     -> ['B']
'throw out everything but the third one'                     -> ['C']
'hard pass on all four'                                      -> []
'A? maybe. dunno.'                                           -> *
'the last two are garbage'                                   -> ['A', 'B']
'B leaves D in the dust'                                     -> ['B']
'A is as good as C but not better than D'                    -> ['A', 'C']
'meh whatever pick anything'                                 -> *
'the second and fourth ones look promising'                  -> ['B', 'D']
'not really feeling any of them tbh'                         -> *
'A is decent, B slightly better, C and D no'                 -> ['A', 'B']
'give me option C or nothing'                                -> ['C']


In [66]:
MY_UTTERANCE = "E is good"
OPTIONS = ["A", "B", "C", "D","E"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: {OPTIONS}\nUser: {MY_UTTERANCE}'},
]
raw = generate_ft(messages)
parsed = parse_output(raw)
print(f"utterance: {MY_UTTERANCE!r}")
print(f"raw:       {raw!r}")
print(f"parsed:    {parsed}")

utterance: 'E is good'
raw:       '["E"]'
parsed:    None


In [67]:
def parse_output(text, valid_letters=None):
    if valid_letters is None:
        valid_letters = ["A","B","C","D"]
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in valid_letters for x in parsed)):
        return parsed
    return None

In [72]:
tests = [
    ("E is good", ["A","B","C","D","E"]),
    ("A and E are best", ["A","B","C","D","E"]),
    ("everything except E is wrong", ["A","B","C","D","E"]),
    ("the fifth one is good", ["A","B","C","D","E"]),
    ("not E, not D", ["A","B","C","D","E"]),
    ("F is the best", ["A","B","C","D","E","F"]),
    ("the last one only", ["A","B","C","D","E","F","G"]),
    ("none of these", ["A","B","C","D","E","F"]),
    ("I don't know", ["A","B","C","D","E","F"]),
    ("only G matters", ["A","B","C","D","E","F","G"]),
    ("the last one only", ["A","B","C","D","E","F","G","H","I","J"]),
    ("the last one is out for sure", ["A","B","C","D","E","F","G","H","I","J"]),
]

for utt, opts in tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {opts}\nUser: {utt}'},
    ]
    raw = generate_ft(messages)
    parsed = parse_output(raw, valid_letters=opts)
    print(f"{utt!r:50s} opts={len(opts)} -> raw={raw!r:20s} parsed={parsed}")

'E is good'                                        opts=5 -> raw='["E"]'              parsed=['E']
'A and E are best'                                 opts=5 -> raw='["A", "E"]'         parsed=['A', 'E']
'everything except E is wrong'                     opts=5 -> raw='["E"]'              parsed=['E']
'the fifth one is good'                            opts=5 -> raw='["E"]'              parsed=['E']
'not E, not D'                                     opts=5 -> raw='["A", "B", "C"]'    parsed=['A', 'B', 'C']
'F is the best'                                    opts=6 -> raw='["F"]'              parsed=['F']
'the last one only'                                opts=7 -> raw='["E", "F", "G"]'    parsed=['E', 'F', 'G']
'none of these'                                    opts=6 -> raw='[]'                 parsed=[]
"I don't know"                                     opts=6 -> raw='"*"'                parsed=*
'only G matters'                                   opts=7 -> raw='["G"]'              parse

In [69]:
MY_UTTERANCE = "Cant tell really but A is better than the rest"
OPTIONS = ["A", "B", "C", "D","E"]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f'Options: {OPTIONS}\nUser: {MY_UTTERANCE}'},
]
raw = generate_ft(messages)
parsed = parse_output(raw)
print(f"utterance: {MY_UTTERANCE!r}")
print(f"raw:       {raw!r}")
print(f"parsed:    {parsed}")

utterance: 'Cant tell really but A is better than the rest'
raw:       '["A"]'
parsed:    ['A']
